# scWAT Xenium slide-level QC summary

Aggregate the four independently processed sections without treating cells as biological replicates.

## Goal

Verify complete Region 1-4 coverage, compare QC distributions and gates, and write Cell-inspired slide-level figures.

## Setup

### Parameters

In [ ]:
PROJECT_ROOT <- "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu"
PIPELINE_REPO <- file.path(PROJECT_ROOT, "adipose_analysis", "YNH_Xenium_scWAT")
RUN_LABEL <- "full_notebook_qc_v1"
EXPECTED_SECTION_COUNT <- 4L


In [ ]:
RUN_ROOT <- file.path(PROJECT_ROOT, "adipose_analysis", "scwat_qc_outputs", RUN_LABEL)
source(file.path(PIPELINE_REPO, "R", "source.R"))
for (package in c("Matrix", "jsonlite", "ggplot2")) require_package(package)
assert_path_within(PROJECT_ROOT, RUN_ROOT)
assert_path_within(PROJECT_ROOT, tempdir())
stopifnot(EXPECTED_SECTION_COUNT == 4L)
cat("Slide QC run root:", RUN_ROOT, "\n")


## Inputs

Exactly four independently completed section bundles are required.

## Completeness Checks

In [ ]:
coverage <- validate_four_section_outputs(RUN_ROOT, paste0("Region_", seq_len(EXPECTED_SECTION_COUNT)))
coverage


## QC Results

In [ ]:
slide_data <- read_slide_qc_outputs(RUN_ROOT, coverage$region_id)
slide_summary <- summarise_slide_qc(slide_data)
slide_summary$section_summary


## Cell-style Figures

Colors are fixed across sections; distributions are descriptive and do not imply cell-level biological replication.

In [ ]:
slide_plots <- plot_slide_qc(slide_data, slide_summary)
for (plot in slide_plots) print(plot)


## Readiness

The worst section gate determines slide readiness. Synthetic metadata and unresolved imaging errors block biology.

In [ ]:
slide_summary$readiness
cat("Overall slide QC status:", slide_summary$overall_status, "\n")


## Outputs

In [ ]:
slide_artifacts <- write_slide_qc_artifacts(PROJECT_ROOT, RUN_ROOT, slide_data, slide_summary, slide_plots)
saved_summary <- readRDS(file.path(RUN_ROOT, "slide_summary", "slide_qc_summary.rds"))
stopifnot(nrow(saved_summary$data$coverage) == 4L)
stopifnot(length(unique(saved_summary$data$cell_metadata$region_id)) == 4L)
data.frame(artifact = basename(slide_artifacts), path = slide_artifacts)
